# Electrodynamics from Scratch — Lecture 12: The Liénard-Wiechert Potentials

In Lecture 11 we saw that changes in a charge's field propagate at the speed of light, so the potential we measure *now* was produced by the charge *earlier* — at the **retarded time** $\tau = t - R/c$, where $R$ is the distance the signal had to travel. Lecture 12 evaluates the retarded potentials $\phi(\mathbf{r}, t)$ and $\mathbf{A}(\mathbf{r}, t)$ *exactly* for a point charge in arbitrary motion. The result is the **Liénard-Wiechert potentials**:

$$
\phi(\mathbf{r}, t) \;=\; \frac{1}{4\pi\varepsilon_0}\, \frac{q}{\left(R - \mathbf{R}\cdot\mathbf{v}/c\right)}
\;=\; \frac{1}{4\pi\varepsilon_0}\, \frac{q}{R\,(1 - \hat{\mathbf{n}}\cdot\mathbf{v}/c)},
\qquad
\mathbf{A}(\mathbf{r}, t) \;=\; \frac{\mathbf{v}}{c^2}\,\phi(\mathbf{r}, t),
$$

where $\mathbf{R} = \mathbf{r} - \mathbf{r}_q(\tau)$ points from the *retarded* position of the charge to the field point, $\hat{\mathbf{n}} = \mathbf{R}/R$, and $\mathbf{v} = \mathbf{v}(\tau)$ is the velocity at the retarded time. The factor $(1 - \hat{\mathbf{n}}\cdot\mathbf{v}/c)$ is the whole point: it is *not* the naive Coulomb potential evaluated at the instantaneous position. The denominator is smallest ahead of the moving charge, so the potential is compressed (stronger) in front of it and stretched (weaker) behind it — the field of a moving charge is beamed forward.

Differentiating the potentials (carefully — both $\mathbf{r}_q(\tau)$ and $\mathbf{v}(\tau)$ depend on the field point through $\tau$) gives the Liénard-Wiechert **fields**:

$$
\mathbf{E} \;=\; \frac{q}{4\pi\varepsilon_0}\, \frac{(1 - \beta^2)(\hat{\mathbf{n}} - \boldsymbol{\beta})}{R^2}
\;+\; \frac{q}{4\pi\varepsilon_0}\, \frac{\hat{\mathbf{n}}\times\left[(\hat{\mathbf{n}} - \boldsymbol{\beta})\times\mathbf{a}\right]}{c\,R},
\qquad
\mathbf{B} \;=\; \frac{1}{c}\, \hat{\mathbf{n}}\times\mathbf{E},
$$

with $\boldsymbol{\beta} = \mathbf{v}/c$. The first (velocity) term falls off as $1/R^2$ and reduces to the Coulomb field when the charge is at rest; it is the "bound" field that stays attached to the charge. The second (acceleration) term falls off as $1/R$ — one power slower — so it dominates at large distance: it is the **radiation field**, carrying energy away to infinity. In this notebook we compute both potentials and fields numerically, split $\mathbf{E}$ into its velocity and radiation pieces, check the relation $\mathbf{A} = \mathbf{v}\,\phi/c^2$, and watch the radiation term beam forward as the charge speed grows.

One subtlety about the numerics: the retarded time itself must be found by solving $|\mathbf{r} - \mathbf{r}_q(\tau)| = c\,(t - \tau)$, because the charge *moves* while the signal travels. For constant velocity this is a quadratic equation in $\tau$, which we solve below and then feed the retarded state into the package's Liénard-Wiechert functions.

In [ ]:
%matplotlib inline
import electrodynamics_from_scratch as eds
import numpy as np
import matplotlib.pyplot as plt

## 1. The potentials of a uniformly moving charge

A charge $q$ starts at the origin at $t = 0$ and slides along $x$ with constant velocity $\mathbf{v} = v\,\hat{\mathbf{x}}$, so $\mathbf{r}_q(\tau) = \mathbf{v}\,\tau$. The retarded-time condition $|\mathbf{r} - \mathbf{v}\tau| = c\,(t-\tau)$ squares to a quadratic,
$$
(v^2 - c^2)\,\tau^2 + 2\,(\mathbf{\Delta}\cdot\mathbf{v} + c^2 t)\,\tau + (\Delta^2 - c^2 t^2) = 0,
\qquad \mathbf{\Delta} = \mathbf{r} - \mathbf{r}_q(0),
$$
of which we keep the root with $\tau < t$ (the retarded, causal one). We then evaluate the Liénard-Wiechert $\phi$ along an observation line offset from the trajectory, and compare it with the naive Coulomb potential of the charge frozen at its $t=0$ position.

In [ ]:
q = eds.ELEMENTARY_CHARGE          # one proton
c = eds.C_LIGHT
v = 0.01 * c                       # constant velocity along x
vel = np.array([v, 0.0, 0.0])
t_obs = 0.0                        # observation time

# Observation line: 300 points offset from the trajectory
x_line = np.linspace(-2e-9, 2e-9, 300)
y_off = 0.5e-9
points = np.column_stack([x_line, np.full_like(x_line, y_off), np.zeros_like(x_line)])

# Solve the retarded-time quadratic for every point (vectorized)
Delta = points - np.zeros(3)                      # r - r_q(0)
A_q = v**2 - c**2
B_q = 2.0 * (Delta @ vel + c**2 * t_obs)
C_q = np.einsum("ij,ij->i", Delta, Delta) - c**2 * t_obs**2
disc = np.sqrt(B_q**2 - 4.0 * A_q * C_q)
tau_roots = np.column_stack([(-B_q + disc) / (2 * A_q), (-B_q - disc) / (2 * A_q)])
tau = np.where(tau_roots[:, 0] < t_obs, tau_roots[:, 0], tau_roots[:, 1])

# Sanity check: the light-cone condition |r - r_q(tau)| = c (t - tau)
residual = np.linalg.norm(points - tau[:, None] * vel[None, :], axis=1) - c * (t_obs - tau)
print("max |retarded-time residual| (m):", np.abs(residual).max())
print("tau range (s): [%.3e, %.3e]" % (tau.min(), tau.max()))

# Lienard-Wiechert potentials: the package expects the RETARDED source
# state, which the caller supplies point by point
phi_lw = np.empty(len(points))
A_vec = np.empty((len(points), 3))
for i, (pt, taui) in enumerate(zip(points, tau)):
    src_ret = vel * taui                     # r_q(tau) = v tau
    phi_lw[i], A_vec[i] = eds.lieard_wiechert_potentials(q, src_ret, vel, pt[None, :], t_obs)

# Naive Coulomb potential of the charge frozen at the t = 0 position
phi_naive, _ = eds.superpose(charges=[(q, np.zeros(3))], points=points)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.plot(x_line * 1e9, phi_lw, label=r"Liénard-Wiechert $\phi$", color="crimson")
ax.plot(x_line * 1e9, phi_naive, label="naive Coulomb (frozen charge)", color="gray", ls="--")
ax.axvline(0, color="k", lw=0.5, alpha=0.5)
ax.set_xlabel("x (nm)   [charge at origin, moving +x]")
ax.set_ylabel(r"$\phi$ (V)")
ax.set_title(r"$\phi$ along an observation line: $(1-\hat{n}\cdot v/c)$ compresses the potential ahead")
ax.legend(loc="best")
fig, ax

## 2. The relation $\mathbf{A} = \mathbf{v}\,\phi/c^2$

The Liénard-Wiechert vector potential is locked to the scalar potential: $\mathbf{A} = \mathbf{v}\,\phi/c^2$. This is a signature of the **Lorenz gauge**, in which the potentials satisfy $\nabla\cdot\mathbf{A} + \frac{1}{c^2}\,\partial\phi/\partial t = 0$ and propagate as wave equations at the speed of light. Because $\mathbf{A}/\phi = \mathbf{v}/c^2$, measuring both potentials directly determines the (retarded) velocity of the source.

In [ ]:
A_predicted = (vel[None, :] / c**2) * phi_lw[:, None]
max_dev = np.abs(A_vec - A_predicted).max()
print("max |A - v phi / c^2| over the line (T m):", max_dev)
print("A_x / phi on the line (should be v/c^2 = %.6e):" % (v / c**2))
print("  measured range: [%.6e, %.6e]" % ((A_vec[:, 0] / phi_lw).min(), (A_vec[:, 0] / phi_lw).max()))

## 3. Velocity and acceleration fields: $1/R^2$ versus $1/R$

Now let the charge accelerate: we evaluate the full Liénard-Wiechert field at points along a radial line, given a fixed retarded state (position at the origin, velocity along $x$, acceleration along $x$). Calling the field function twice — once with the true acceleration and once with $\mathbf{a}=0$ — isolates the velocity piece; subtracting gives the radiation piece via `eds.split_velocity_acceleration`. On a log-log plot the two terms appear as straight lines with slopes $-2$ and $-1$: the radiation term wins at large distance, which is why accelerated charges radiate to infinity while bound fields do not.

In [ ]:
beta = 0.1
vel_f = beta * c * np.array([1.0, 0.0, 0.0])
acc_f = np.array([1.0e18, 0.0, 0.0])          # acceleration along x (m/s^2)
src_f = np.zeros(3)

# Radial observation line at 45 degrees in the xy plane
n_hat = np.array([np.cos(np.pi / 4), np.sin(np.pi / 4), 0.0])
R_vals = np.linspace(0.5e-9, 5e-9, 100)
radial_points = src_f + R_vals[:, None] * n_hat[None, :]

E_total, B_total = eds.lieard_wiechert_fields(q, src_f, vel_f, acc_f, radial_points)
E_vel_only, _ = eds.lieard_wiechert_fields(q, src_f, vel_f, np.zeros(3), radial_points)
E_vel, E_acc = eds.split_velocity_acceleration(E_total, E_vel_only)

fig, ax = eds.plot_decay(
    R_vals,
    {
        r"$|E_{\rm velocity}|$ ($\propto 1/R^2$)": np.linalg.norm(E_vel, axis=1),
        r"$|E_{\rm acceleration}|$ ($\propto 1/R$)": np.linalg.norm(E_acc, axis=1),
    },
    title="Velocity vs acceleration field along a radial line",
)
fig, ax

amp_v, exp_v = eds.multipole.power_law_fit(R_vals, np.linalg.norm(E_vel, axis=1))
amp_a, exp_a = eds.multipole.power_law_fit(R_vals, np.linalg.norm(E_acc, axis=1))
print("fitted exponent of the velocity term:     %.3f  (expect -2)" % exp_v)
print("fitted exponent of the acceleration term: %.3f  (expect -1)" % exp_a)
print("crossover radius where |E_acc| = |E_vel|: %.2e m" % (amp_a / amp_v if exp_a > exp_v else float("nan")))

## 4. Angular beaming of the radiation term

For a charge accelerating perpendicular to its velocity, the radiation term is roughly dipolar ($\propto \sin^2\theta$) when the charge is slow — but as $\beta$ grows the factor $(\hat{\mathbf{n}} - \boldsymbol{\beta})$ inside the cross product tilts the pattern *forward*, into the direction of motion. We sample $|\mathbf{E}_{\rm acc}|$ on a circle of fixed $R$ around the retarded position for $\beta = 0.05$ (nearly non-relativistic) and $\beta = 0.5$ (relativistic), normalizing each pattern to its peak.

In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 181)
R_beam = 1.0e-9
beam_points = R_beam * np.column_stack([np.cos(theta), np.sin(theta), np.zeros_like(theta)])
acc_perp = np.array([0.0, 1.0e18, 0.0])       # a perpendicular to v (along y)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for beta_b, color in [(0.05, "gray"), (0.5, "crimson")]:
    vel_b = beta_b * c * np.array([1.0, 0.0, 0.0])
    E_tot, _ = eds.lieard_wiechert_fields(q, np.zeros(3), vel_b, acc_perp, beam_points)
    E_v0, _ = eds.lieard_wiechert_fields(q, np.zeros(3), vel_b, np.zeros(3), beam_points)
    _, E_rad = eds.split_velocity_acceleration(E_tot, E_v0)
    mag = np.linalg.norm(E_rad, axis=1)
    mag /= mag.max()
    ax.plot(np.degrees(theta), mag, color=color, label=r"$\beta = %.2f$" % beta_b)

ax.axvline(0, color="k", lw=0.5, alpha=0.4)
ax.set_xlabel(r"$\theta$ (deg) measured from the velocity $\hat{x}$ (0$^\circ$ = forward)")
ax.set_ylabel(r"normalized $|\mathbf{E}_{\rm acc}|$")
ax.set_title("Radiation beaming: the pattern tilts forward as $\\beta$ grows")
ax.legend(loc="best")
fig, ax

## 5. Structure of the radiation field: transverse, with $\mathbf{B} = \hat{\mathbf{n}}\times\mathbf{E}/c$

Two structural facts make the acceleration term a genuine radiation field. First, it is **transverse**: $\mathbf{E}_{\rm acc}\cdot\hat{\mathbf{n}} = 0$, so the field shakes the detector sideways rather than pushing along the line of sight. Second, it travels with a magnetic partner of fixed ratio: $\mathbf{B} = \hat{\mathbf{n}}\times\mathbf{E}/c$, the same relation carried by a plane wave, with $|\mathbf{B}| = |\mathbf{E}|/c$. We verify both numerically on the beaming circle.

In [ ]:
vel_c = 0.5 * c * np.array([1.0, 0.0, 0.0])
E_c, B_c = eds.lieard_wiechert_fields(q, np.zeros(3), vel_c, acc_perp, beam_points)
E_c0, _ = eds.lieard_wiechert_fields(q, np.zeros(3), vel_c, np.zeros(3), beam_points)
_, E_rad_c = eds.split_velocity_acceleration(E_c, E_c0)

n_circ = beam_points / R_beam                      # unit vectors on the circle
transverse = np.abs(np.einsum("ij,ij->i", E_rad_c, n_circ))
B_expected = np.cross(n_circ, E_c) / c

print("max |E_acc . n| / |E_acc|  (expect ~0):", (transverse / np.linalg.norm(E_rad_c, axis=1)).max())
print("max |B - n x E / c| (T):               ", np.abs(B_c - B_expected).max())
print("max | |B| - |E|/c |  (relative):       ", np.abs(np.linalg.norm(B_c, axis=1) / (np.linalg.norm(E_c, axis=1) / c) - 1.0).max())

## Exercises

1. **Conceptual.** In the Liénard-Wiechert potential $\phi = \frac{q}{4\pi\varepsilon_0}\,\frac{1}{R(1-\hat{\mathbf{n}}\cdot\mathbf{v}/c)}$, what physically causes the factor $(1 - \hat{\mathbf{n}}\cdot\mathbf{v}/c)$? What happens to $\phi$ ahead of the charge as $v \to c$, and why can $(1 - \hat{\mathbf{n}}\cdot\mathbf{v}/c)$ never vanish for a massive charge?

2. **Coding.** In the Section 1 cell, change the velocity to $v = 0.1\,c$ and the observation time to $t = 10^{-9}$ s (shift the charge's $t=0$ position accordingly, e.g. start it at $x_0 = -v t$ so it stays near the origin). Plot the Liénard-Wiechert $\phi$ against the naive Coulomb $\phi$ again and describe how the forward compression changes.

3. **Conceptual.** Explain why the acceleration term of $\mathbf{E}$ falls off as $1/R$ while the velocity term falls off as $1/R^2$, and why this single power difference means that *any* energy radiated by an accelerated charge escapes to infinity while the bound field does not.

4. **Coding.** Using `eds.lieard_wiechert_fields` with $\beta = 0.9$ and $\mathbf{a} \perp \mathbf{v}$, compute $|\mathbf{E}_{\rm acc}|$ on a circle as in Section 4 (use `eds.split_velocity_acceleration` with an $\mathbf{a}=0$ run to isolate it). Estimate the half-angle of the forward-beaming cone inside which the pattern exceeds half its peak, and compare with the estimate $\Delta\theta \sim 1/\gamma$ from the book.

5. **Conceptual.** The relation $\mathbf{A} = \mathbf{v}\,\phi/c^2$ holds for the Liénard-Wiechert potentials. Which gauge condition do the potentials satisfy, how is it written in terms of $\phi$ and $\mathbf{A}$, and why is that gauge the natural one for radiation problems? (You can explore gauge changes numerically with `eds.gauge.gauge_transform` and `eds.gauge.check_gauge_invariance` on a static configuration.)

6. **Coding.** Use `eds.waves.larmor_power` to compute the total power radiated by the accelerated charge of Section 3 ($q$ = one proton, $a = 10^{18}\,\mathrm{m/s^2}$), then rescale the result for an electron with the same acceleration and explain the difference in terms of the $q^2$ and $m$ dependence of radiated power for a given applied force.